# Zeno Neural Volume — execução real dos nodes + inspeção gráfica

Este notebook compila o runtime **headless real do Zeno**, executa os nodes `ReadVDB`, `ExportNeuralVolumeDataset` e `ZNVDSFileToNanoVDB` em um grafo Zeno, e mostra visualizações do volume exportado. A execução é CPU-only; não simula treinamento neural nem inferência CUDA.

Use um runtime Colab com RAM suficiente. A compilação pode levar vários minutos. A fixture é criada por um executável separado, gravada em `.vdb` e depois lida pelo node `ReadVDB`.

In [ ]:
import os, subprocess, pathlib, json, struct, math, sys
ROOT = pathlib.Path('/content/zeno')
WORK = pathlib.Path('/content/zeno-node-visual-test')
WORK.mkdir(exist_ok=True)
print('CPUs:', os.cpu_count())
try:
    import torch
    print('CUDA disponível no Colab:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch/GPU não consultado:', exc)


In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cmake ninja-build g++ pkg-config git \
  libopenvdb-dev libtbb-dev libblosc-dev zlib1g-dev \
  libboost-iostreams-dev libboost-system-dev libboost-filesystem-dev \
  libboost-regex-dev libimath-dev libopenexr-dev


In [ ]:
%cd /content
!git clone --depth 1 --branch zeno-master-clean-copy https://github.com/brucelee1985666-commits/zeno.git zeno
%cd /content/zeno
!git submodule update --init projects/zenvdb/nanovdb-upstream
!git rev-parse HEAD


In [ ]:
%cd /content/zeno
!cmake -S . -B build-zeno-node-runtime -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DZENO_BUILD_EDITOR=OFF -DZENO_BUILD_DESIGNER=OFF -DZENO_BUILD_PLAYER=OFF \
  -DZENO_WITH_zenvdb=ON -DZENO_SYSTEM_OPENVDB=ON \
  -DZENO_BUILD_NEURAL_VOLUME_NODE_TESTS=ON -DZENO_WITH_CUDA=OFF \
  -DZENO_USE_CCACHE=OFF -DZENO_ENABLE_OPENMP=OFF
!cmake --build build-zeno-node-runtime --target zeno-neural-volume-node-smoke write-independent-vdb-fixture --parallel 2


In [ ]:
VDB = WORK / 'independent-input.vdb'
ZNVDS = WORK / 'from-zeno-nodes.znvds'
NVDB = WORK / 'from-zeno-nodes.nvdb'
!/content/zeno/build-zeno-node-runtime/bin/write-independent-vdb-fixture "$VDB"
!/content/zeno/build-zeno-node-runtime/bin/zeno-neural-volume-node-smoke "$VDB" "$ZNVDS" "$NVDB"
!python /content/zeno/tools/neural_volume/inspect_znvds.py "$ZNVDS" --domain-size 128
print('Arquivos:', {p.name: p.stat().st_size for p in [VDB, ZNVDS, NVDB]})


## Visualização do ZNVDS produzido pelos nodes

A visualização abaixo decodifica os voxels ativos do arquivo `.znvds` gerado pelo node. O mapa 2D mostra a fatia `z=0`; o gráfico 3D mostra a distribuição espacial e a densidade. Os tiles internos são listados separadamente para não expandir regiões enormes em memória.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
data = ZNVDS.read_bytes(); pos = 0
def take(n):
    global pos
    out = data[pos:pos+n]
    if len(out) != n: raise ValueError('ZNVDS truncado durante visualização')
    pos += n
    return out
def u32(): return struct.unpack('<I', take(4))[0]
def u64(): return struct.unpack('<Q', take(8))[0]
def i32(): return struct.unpack('<i', take(4))[0]
def f32(): return struct.unpack('<f', take(4))[0]
assert take(8) == b'ZNVDS001'
version, flags = u32(), u32()
leaf_count, tile_count, active_count = u64(), u64(), u64()
name = take(u32()).decode('utf-8')
grid_class, background = u32(), f32()
voxel_size = [f32() for _ in range(3)]
affine = [struct.unpack('<d', take(8))[0] for _ in range(16)]
tiles = []
for _ in range(tile_count):
    xyz = (i32(), i32(), i32()); level = u32(); value = f32()
    tiles.append((xyz, level, value))
points = []
for _ in range(leaf_count):
    ox, oy, oz = i32(), i32(), i32(); count = u32(); mask = take(64)
    for _ in range(count):
        offset = struct.unpack('<H', take(2))[0]; value = f32()
        x = ox + (offset >> 6); y = oy + ((offset >> 3) & 7); z = oz + (offset & 7)
        points.append((x,y,z,value))
assert len(points) == active_count, (len(points), active_count)
arr = np.full((33,33), np.nan, dtype=np.float32)
for x,y,z,v in points:
    if z == 0 and -16 <= x <= 16 and -16 <= y <= 16:
        arr[y+16, x+16] = v
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(arr, origin='lower', extent=[-16.5,16.5,-16.5,16.5], cmap='inferno')
ax.set_title(f'Zeno node output — density slice z=0 ({name})'); ax.set_xlabel('X'); ax.set_ylabel('Y')
fig.colorbar(im, ax=ax, label='FloatGrid density'); plt.show()
sample = points if len(points) <= 12000 else points[::max(1, len(points)//12000)]
xyz = np.asarray([[p[0],p[1],p[2]] for p in sample]); vals = np.asarray([p[3] for p in sample])
fig = plt.figure(figsize=(7,6)); ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=vals, s=4, cmap='inferno')
ax.set_title('Active voxels from Zeno-generated ZNVDS'); ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
fig.colorbar(sc, ax=ax, shrink=0.65, label='Density'); plt.show()
print('Grid:', name, '| active voxels:', len(points), '| active tiles:', tiles)


In [ ]:
from google.colab import files
for p in [VDB, ZNVDS, NVDB]: files.download(str(p))


## Como interpretar o resultado

- `ZENO NODE GRAPH PASS` confirma que o grafo executou os nodes registrados no runtime Zeno, não apenas funções auxiliares isoladas.
- A comparação verifica valores, ocupação, nome/classe e transformação após reabrir os arquivos.
- Os gráficos são inspeção visual CPU do dataset, não um render volumétrico final do Zeno/OptiX.
- A compilação CUDA e qualquer treino/inferência neural são etapas distintas. Este notebook não as declara aprovadas.